# A/B Test Design — Future Campaign Planning

**Context**: After discovering the Q4 2025 campaign wasn't randomly assigned, leadership asked: *"How should we design the Q1 2026 campaign so we don't need these corrections?"*

This notebook designs a proper randomized experiment for the next campaign, including:
1. Power analysis (how many stores do we need?)
2. Statistical test methodology
3. Sequential testing for early stopping
4. Multiple comparison correction for testing several KPIs

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from causal.ab_testing import (
    calculate_sample_size, analyze_ab_test,
    bonferroni_correction, sequential_test
)
from data.data_loader import load_panel_data

panel = load_panel_data()

# Baseline stats from control stores (our best estimate of "no promotion" behavior)
control = panel[panel['treated'] == 0]
baseline_revenue = control['revenue'].mean()
baseline_std = control['revenue'].std()
print(f'Baseline (control stores): mean=${baseline_revenue:,.0f}/week, std=${baseline_std:,.0f}')

## 1. Power Analysis — How Many Stores?

From the Q4 analysis, we estimate the true promotion effect is ~8%. But what if Q1 is weaker? We should power the test for a smaller effect to be safe.

In [ ]:
print('Sample Size Requirements for Q1 2026 Campaign')
print(f'(alpha=0.05, power=0.80, baseline=${baseline_revenue:,.0f}/wk)')
print('-' * 55)
print(f'{"MDE":>6}  {"Stores/Group":>14}  {"Total Stores":>14}  Note')
print('-' * 55)
for mde in [0.02, 0.03, 0.05, 0.08, 0.10]:
    pa = calculate_sample_size(baseline_revenue, baseline_std, mde=mde)
    total = pa.required_sample_per_group * 2
    feasible = 'Feasible' if total <= 500 else 'Need more stores'
    print(f'{mde:>5.0%}  {pa.required_sample_per_group:>14,}  {total:>14,}  {feasible}')

print(f'\nRecommendation: Target MDE=5% (conservative). Need ~{calculate_sample_size(baseline_revenue, baseline_std, 0.05).required_sample_per_group * 2} stores total.')
print('With 500 stores available, we have enough for 250/group.')

## 2. Analyzing the Q4 Data as if It Were an A/B Test

For comparison: what does a standard t-test say? (This is biased because assignment wasn't random, but useful as a reference.)

In [ ]:
# Post-period only (promo weeks)
post = panel[panel['post_period'] == 1]
control_rev = post[post['treated'] == 0]['revenue'].values
treated_rev = post[post['treated'] == 1]['revenue'].values

result = analyze_ab_test(control_rev, treated_rev)

print('=== A/B Test Result (Post-Period Only) ===')
print(f'Control mean:    ${result.control_mean:,.2f}')
print(f'Treatment mean:  ${result.treatment_mean:,.2f}')
print(f'Absolute lift:   ${result.absolute_lift:,.2f}')
print(f'Relative lift:   {result.relative_lift:.2%}')
print(f'p-value:         {result.p_value:.6f}')
print(f'95% CI:          [${result.ci_lower:,.2f}, ${result.ci_upper:,.2f}]')
print(f'Significant:     {result.significant}')
print()
print('Note: This test is biased (non-random assignment). Use PSM/DiD/IV for causal estimates.')

## 3. Sequential Testing (SPRT)

For Q1 2026: if we run a proper randomized test, we can check results weekly using SPRT and stop early if the answer is clear. This saves money.

In [ ]:
# Simulate: what if we collected data weekly from a real randomized test?
rng = np.random.default_rng(42)
simulated_treatment = rng.normal(baseline_revenue * 1.08, baseline_std, 500).tolist()

sprt = sequential_test(simulated_treatment, control_mean=baseline_revenue, control_std=baseline_std)

print('=== Sequential Testing Simulation ===')
print(f'Decision:              {sprt["decision"].replace("_", " ").title()}')
print(f'Observations needed:   {sprt["stopped_at"]} / 500')
print(f'Savings:               {(1 - sprt["stopped_at"]/500):.0%} fewer observations needed')
print()
print('With SPRT, we can detect an 8% effect early and stop the test,')
print('freeing up budget for the full rollout sooner.')

## 4. Multiple Comparison Correction

Finance wants us to test 4 KPIs simultaneously. Without correction, we inflate the false positive rate to ~19%. Bonferroni keeps it at 5%.

In [ ]:
# Simulated p-values from testing 4 KPIs
kpi_names = ['Revenue Lift', 'Units Sold', 'Basket Size', 'New Customers']
raw_p = [0.008, 0.032, 0.048, 0.061]

corrected = bonferroni_correction(raw_p)
corrected.index = kpi_names

print('=== Multiple Comparison Correction (Bonferroni) ===')
print(corrected.to_string())
print()
print('After correction, only Revenue Lift and Units Sold remain significant.')
print('Basket Size (p=0.048) loses significance — we should not claim a basket effect.')